
# Assignment 1: Boolean Model, TF-IDF, and Data Retrieval vs. Information Retrieval Conceptual Questions

**Student names**: _Your_names_here_ <br>
**Group number**: _Your_group_here_ <br>
**Date**: _Submission Date_

## Important notes
Please carefully read the following notes and consider them for the assignment delivery. Submissions that do not fulfill these requirements will not be assessed and should be submitted again.
1. You may work in groups of maximum 2 students.
2. The assignment must be delivered in ipynb format.
3. The assignment must be typed. Handwritten assignments are not accepted.

**Due date**: 18.09.2026 23:59

In this assignment, you will:
- Implement a Boolean retrieval model
- Compute TF-IDF vectors for documents
- Run retrieval on queries
- Answer conceptual questions 

---
## Dataset

You will use the **Cranfield** dataset, provided in this file:

- `cran.all.1400`: The document collection (1400 documents)

**The code to parse the file is ready — just update the cran file path to match your own file location. Use the docs variable in your code for the parsed file**

### Load and parse documents (provided)

Run the cell to parse the Cranfield documents. Update the path so it points to your `cran.all.1400` file.


In [2]:

# Read 'cran.all.1400' and parse the documents into a suitable data structure

CRAN_PATH = "./cran.all.1400"  # <-- change this!

def parse_cranfield(path):
    docs = {}
    current_id = None
    current_field = None
    buffers = {"T": [], "A": [], "B": [], "W": []}
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith(".I "):
                if current_id is not None:
                    docs[current_id] = {
                        "id": current_id,
                        "title": " ".join(buffers["T"]).strip(),
                        "abstract": " ".join(buffers["W"]).strip()
                    }
                current_id = int(line.split()[1])
                buffers = {k: [] for k in buffers}
                current_field = None
            elif line.startswith("."):
                tag = line[1:].strip()
                current_field = tag if tag in buffers else None
            else:
                if current_field is not None:
                    buffers[current_field].append(line)
    if current_id is not None:
        docs[current_id] = {
            "id": current_id,
            "title": " ".join(buffers["T"]).strip(),
            "abstract": " ".join(buffers["W"]).strip()
        }
    print(f"Parsed {len(docs)} documents.")
    return docs

docs = parse_cranfield(CRAN_PATH)



Parsed 1400 documents.


## 1.1 – Boolean Retrieval Model

### 1.1.1 Tokenize documents

Implement tokenization using the given list of stopwords. Create a list of normalized terms per document (e.g., lowercase, remove punctuation/digits; drop stopwords). Store the token lists to use in later steps.

In [3]:
# TODO: Implement tokenization using the given list of stopwords, create list of terms per document



STOPWORDS = set("""a about above after again against all am an and any are aren't as at be because been
before being below between both but by can't cannot could couldn't did didn't do does doesn't doing don't down
during each few for from further had hadn't has hasn't have haven't having he he'd he'll he's her here here's hers
herself him himself his how how's i i'd i'll i'm i've if in into is isn't it it's its itself let's me more most
mustn't my myself no nor not of off on once only or other ought our ours ourselves out over own same shan't she
she'd she'll she's should shouldn't so some such than that that's the their theirs them themselves then there there's
these they they'd they'll they're they've this those through to too under until up very was wasn't we we'd we'll we're
we've were weren't what what's when when's where where's which while who who's whom why why's with won't would wouldn't
you you'd you'll you're you've your yours yourself yourselves""".split())

"""
Docs:
{
    1: {
        "id": 1,
        "title": "experimental investigation of the aerodynamics of a wing in slipstream .",
        "abstract": "an experimental study of a wing in a propeller slipstream was made in order to determine the spanwise distribution of the lift ...",
    },
    2: {
        "id": 2,
        "title": "simple shear flow past a flat plate in an incompressible fluid of small viscosity .",
        "abstract": "in the study of high-speed viscous flow past a two-dimensional body it is usually necessary to consider ...",
    },
    # ... up to 1400
}

"""
# Your code here
# get abstract
# split(" "), then remove all sublists with "." and stopwords

import re

WORD_RE = re.compile(r"[a-z]+(?:-[a-z]+)*")   # letter runs, hyphen-joined allowed

def tokenize(text):
    tokens = []
    for w in WORD_RE.findall(text.lower()):
        if len(w) < 2:            # drop single letters (math variables)
            continue
        if w in STOPWORDS:
            continue
        tokens.append(w)
    return tokens

for doc in docs:
    inner_dict = docs[doc]
    text = inner_dict["abstract"]
    terms = tokenize(text)
        
    inner_dict["terms"] = list(set(terms))
    
for key, val in docs[1].items():
    print(f"{key}:{val}")
                
        
        
            
        
       
    
    
        


id:1
title:experimental investigation of the aerodynamics of a wing in a slipstream .
abstract:experimental investigation of the aerodynamics of a wing in a slipstream .   an experimental study of a wing in a propeller slipstream was made in order to determine the spanwise distribution of the lift increase due to slipstream at different angles of attack of the wing and at different free stream to slipstream velocity ratios .  the results were intended in part as an evaluation basis for different theoretical treatments of this problem .   the comparative span loading curves, together with supporting evidence, showed that a substantial part of the lift increment produced by the slipstream was due to a /destalling/ or boundary-layer-control effect .  the integrated remaining lift increment, after subtracting this destalling lift, was found to agree well with a potential flow theory .   an empirical evaluation of the destalling effects was made for the specific configuration of the experim

### Build vocabulary

Create a set (or list) of unique terms from all tokenized documents. Report the number of unique terms.


In [4]:
# TODO: Create a set or list of unique terms

# Report: 
# - Number of unique terms

# Your code here

unique_terms = set()

for doc in docs:
    inner_dict = docs[doc]
    terms = inner_dict["terms"]
    for term in terms:
        unique_terms.add(term)
        
print(list(unique_terms))
print(f"num of unque words: {len(unique_terms)}")

list_sorted = sorted(unique_terms)
term_index = {t: i for i, t in enumerate(list_sorted)}

for key, val in term_index.items():
    print(f"{key}:{val}")
    

['numberical', 'luminous', 'always', 'draw', 'definitions', 'novel', 'carter', 'recorded', 'probability', 'believes', 'detachment', 'advance', 'large-chord', 'protecting', 'demonstrated', 'inviscid', 'aerothermodynamic', 'powell', 'translation', 'overriding', 'answer', 'vessel', 'reaches', 'reformulated', 'transformations', 'proceeding', 'now', 'fixing', 'subcritical', 'plate-glass', 'perfect', 'non-symmetric', 'echoes', 'majority', 'walls', 'unsolved', 'mach-number', 'turbances', 'extents', 'quenching', 'hyperbolic', 'disturbances', 'viscosity-temperature', 'dependency', 'schlichting', 'afford', 'converting', 'shaft-mounted', 'simulates', 'generated', 'holder', 'assumption', 'articles', 'pitot', 'constant-density', 'constant-property', 'circulatory', 'structural-loads', 'popularly', 'multiple-nozzle', 'amounts', 'transonic-tunnel', 'shercliff', 'apex-angles', 'vanishing', 'proportionally', 'land', 'secure', 'spiked-nose', 'distinctly', 'intense', 'seasons', 'body', 'depletion', 'encom

### Build inverted index

For each term, store the list (or set) of document IDs where the term appears.


In [5]:

# TODO: For each term, store list of document IDs where the term appears
# Your code here

inverted_index = {}
for doc_id in docs:
    terms = docs[doc_id]["terms"]
    for term in terms:
        if term in inverted_index:
            inverted_index[term].append(doc_id)
        else:
            inverted_index[term] = [doc_id]
            
            
    






### Retrieve documents for a Boolean query (AND/OR)

Create a function to retrieve documents for a Boolean query (AND/OR) with query terms.  


In [6]:
# TODO: Create a function for retrieving documents for a Boolean query (AND/OR) with query terms

def boolean_retrieve(query:str): 
# Your code here
    raise NotImplementedError("Implement Boolean retrieval here")


In [7]:
# Do not change this code
boolean_queries = [
  "gas AND pressure",
  "structural AND aeroelastic AND flight AND high AND speed OR aircraft",
  "heat AND conduction AND composite AND slabs",
  "boundary AND layer AND control",
  "compressible AND flow AND nozzle",
  "combustion AND chamber AND injection",
  "laminar AND turbulent AND transition",
  "fatigue AND crack AND growth",
  "wing AND tip AND vortices",
  "propulsion AND efficiency"
]

In [8]:
# Run Boolean queries in batch, using the function you created
def run_batch_boolean(queries):
    results = {}
    for i, q in enumerate(queries, 1):
        res = boolean_retrieve(q)
        results[f"Q{i}"] = res
    return results

boolean_results = run_batch_boolean(boolean_queries)
for qid, res in boolean_results.items():
    print(qid, "=>", res[:5])


NotImplementedError: Implement Boolean retrieval here

## Part 1.2 – TF-IDF Indexing


$tf_{i,j} = \text{Raw Frequency}$

$idf_t = \log\left(\frac{N}{df_t}\right)$

### Build document–term matrix (TF and IDF weights)

Compute tf and idf using the formulas above and store the weights in a document–term matrix (rows = documents, columns = terms).



In [9]:
# TODO: Calculate the weights for the documents and the terms using tf and idf weighting. Put these values into a document–term matrix (rows = documents, columns = terms).

# Your code here
import math

N = len(docs)

def term_freq(doc_id):
    tokens = tokenize(docs[doc_id]["abstract"])
    freq = {}
    for t in tokens:
        freq[t] = freq.get(t, 0) + 1
    return freq

doc_term_freq = {doc_id: term_freq(doc_id) for doc_id in docs}

# idf_t = log(N / df_t), df_t from the inverted index built earlier
idf = {term: math.log(N / len(doc_ids)) for term, doc_ids in inverted_index.items()}

tfidf_matrix = {
    doc_id: {term: tf * idf[term] for term, tf in freqs.items()}
    for doc_id, freqs in doc_term_freq.items()
}

print(f"Built TF-IDF matrix for {len(tfidf_matrix)} documents over {len(idf)} terms.")


Built TF-IDF matrix for 1400 documents over 8206 terms.


### Build TF–IDF document vectors

From the matrix, build a TF–IDF vector for each document (consider normalization if needed for cosine similarity).


In [10]:

# TODO: Build TF–IDF document vectors from the document–term matrix
# Your code here

doc_vectors = tfidf_matrix

doc_norms = {}
for doc_id, weights in doc_vectors.items():
    norm = math.sqrt(sum(w * w for w in weights.values()))
    doc_norms[doc_id] = norm if norm > 0 else 1e-12

print("Example vector (doc 1, first 5 terms):", list(doc_vectors[1].items())[:5])


Example vector (doc 1, first 5 terms): [('experimental', 2.9769706040328754), ('investigation', 1.8689491079191851), ('aerodynamics', 4.1087332996742), ('wing', 6.342986402040829), ('slipstream', 23.79660432907675)]


### Implement cosine similarity

Implement a function to compute cosine similarity scores between a (tokenized) query and all documents.


In [11]:

# TODO: Create a function for calculating the similarity score of all the documents by their relevance to query terms

def tfidf_retrieve(query: str):
    q_tokens = tokenize(query)
    q_freq = {}
    for t in q_tokens:
        q_freq[t] = q_freq.get(t, 0) + 1

    q_weights = {t: f * idf[t] for t, f in q_freq.items() if t in idf}
    q_norm = math.sqrt(sum(w * w for w in q_weights.values()))
    if q_norm == 0:
        return []

    scores = []
    for doc_id, weights in doc_vectors.items():
        dot = sum(qw * weights[t] for t, qw in q_weights.items() if t in weights)
        if dot == 0.0:
            continue
        sim = dot / (q_norm * doc_norms[doc_id])
        scores.append((doc_id, sim))

    scores.sort(key=lambda x: x[1], reverse=True)
    return scores


In [12]:
# Do not change this code
tfidf_queries = [
  "gas pressure",
  "structural aeroelastic flight high speed aircraft",
  "heat conduction composite slabs",
  "boundary layer control",
  "compressible flow nozzle",
  "combustion chamber injection",
  "laminar turbulent transition",
  "fatigue crack growth",
  "wing tip vortices",
  "propulsion efficiency"
]

In [13]:
# Run TF-IDF queries in batch (print top-5 results for each), using the function you created
def run_batch_tfidf(queries):
    results = {}
    for i, q in enumerate(queries, 1):
        res = tfidf_retrieve(q)
        results[f"Q{i}"] = res
    return results

tfidf_results = run_batch_tfidf(tfidf_queries)

for qid, res in tfidf_results.items():
    print(qid, "=>", res[:5])


Q1 => [(169, 0.3204987129023791), (167, 0.26227436018773576), (1312, 0.2611422481029525), (1286, 0.25467657214331385), (166, 0.2540679247847052)]
Q2 => [(12, 0.4259404584776456), (51, 0.3153677546508331), (884, 0.25772878979192865), (746, 0.20837065976812172), (1169, 0.19888004908191378)]
Q3 => [(5, 0.4778710066669159), (485, 0.42002825404558386), (399, 0.3919618647137929), (144, 0.3671986541915014), (181, 0.29442863331453756)]
Q4 => [(368, 0.3725216990830361), (748, 0.3514601056378009), (638, 0.33884043322717133), (451, 0.2875920095262341), (1163, 0.2802988154312777)]
Q5 => [(1187, 0.3118623710491635), (970, 0.29779642188448824), (389, 0.28081429017305704), (173, 0.267672348875205), (965, 0.2566182651868475)]
Q6 => [(628, 0.24486360014086944), (125, 0.22853049110393606), (974, 0.22793102291794515), (635, 0.21840816254516585), (1241, 0.21839338673549732)]
Q7 => [(418, 0.45200139289821106), (272, 0.36987228634472513), (315, 0.36702365531710873), (1264, 0.34043444214462765), (9, 0.337111


## Part 1.3 – Conceptual Questions

Answer the following questions:

**1. What is the difference between data retrieval and information retrieval?**
*Your answer here*

**For the following scenarios, which approach would be suitable data retrieval or information retrieval? Explain your reasoning.** <br>
1.a A clerk in pharmacy uses the following query: Medicine_name = Ibuprofen_400mg
- Data retrieval would be better as the query is very datadriven, similar to a database query

1.b A clerk in pharmacy uses the following query: An anti-biotic medicine 
*Your answer here*
- IR query, Natural text language, 

1.c Searching for the schedule of a flight using the following query: Flight_ID = ZEFV2
*Your answer here*
- Data retrieval, easy for a database system to match

1.d Searching an E-commerce website using the following query to find an specific shoe: Brooks Ghost 15
*Your answer here*
- Data retrieval

1.e Searching the same E-commerce website using the following query: Nice running shoes
*Your answer here*

- IR
